# ☢️ SPECT/CT 기반 3D 병변 자동 분할 대시보드 (One-Cell Pipeline)
이 노트북은 임상의의 시각적 검증 및 OpenGATE 선량평가 연동을 위한 대시보드입니다.  
**아래의 셀을 클릭하고 `Shift + Enter`를 한 번만 누르시면 모든 과정이 자동으로 실행됩니다.**

In [ ]:
import SimpleITK as sitk
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from skimage.transform import iradon, radon
from scipy.ndimage import gaussian_filter, shift
import os, warnings

# ==============================================================================
# [임상 검증 모드 설정]
# ==============================================================================
warnings.filterwarnings("ignore")
plt.style.use('default')
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'text.color': '#333333', 'axes.labelcolor': '#333333',
    'xtick.color': '#333333', 'ytick.color': '#333333',
    'axes.edgecolor': '#AAAAAA'
})

print(">>> Initializing Clinical SPECT/CT Volumetry & Dosimetry Dashboard...")

# ==============================================================================
# 1. Configuration (환경 설정 및 물리적 파라미터)
# ==============================================================================
CFG = {
    'SPECT_PATH': './data/SPECT/SPECT-CT_SC001_DS.dcm', 
    'EXPORT_DIR': './Experiments/Thyroid_Recon_Paper',
    
    # [스캐너 기하학 및 재구성 파라미터]
    'ANGLES': 60, 'ARC': 360.0,
    'FBP_FILTER': 'shepp-logan',
    'OSEM_ITER': 4,
    'MAP_ITER': 15, 'MAP_BETA': 0.02,
    
    # [공간 보정 파라미터]
    'FLIP_Z': True, 'FLIP_Y': True,
    'COR_OFFSET': 0.0,    # Center of Rotation 보정 (필요시 -1.5 등 픽셀 단위 입력)
    'SCAN_DIR': 'CCW'     # 스캐너 회전 방향 ('CW' 또는 'CCW')
}

os.makedirs(CFG['EXPORT_DIR'], exist_ok=True)
ALGO_NAMES = ['FBP', 'OSEM', 'MAP']
CMAPS = ['Greys', 'Blues', 'Reds']

# ==============================================================================
# 2. Physics & Reconstruction Engine
# ==============================================================================
class ReconStudio:
    def __init__(self):
        self.vols = {}; self.meta = {}; self.center = (0,0,0)
        self._run_engine()
        
    def _run_engine(self):
        if not os.path.exists(CFG['SPECT_PATH']):
            print(f"🚨 Error: DICOM 파일을 찾을 수 없습니다. 경로를 확인하세요: {CFG['SPECT_PATH']}")
            return

        img = sitk.ReadImage(CFG['SPECT_PATH'])
        raw = sitk.GetArrayFromImage(img).astype(np.float32)
        
        # 차원 정렬
        if raw.shape[0] == CFG['ANGLES']: self.sinograms = np.transpose(raw, (1, 0, 2))
        else: self.sinograms = raw
        
        sp = img.GetSpacing()
        vox_ml = (sp[0] * sp[0] * sp[1]) / 1000.0 
        self.meta = {'spacing': (sp[0], sp[0], sp[1]), 'shape': (self.sinograms.shape[0], self.sinograms.shape[2], self.sinograms.shape[2]), 'vox_ml': vox_ml}
        
        d, angles, w = self.sinograms.shape
        
        # 회전 방향 보정
        if CFG['SCAN_DIR'] == 'CW':
            theta = np.linspace(0., CFG['ARC'], angles, endpoint=False)[::-1]
        else:
            theta = np.linspace(0., CFG['ARC'], angles, endpoint=False)
            
        v_fbp, v_osem, v_map = [np.zeros((d, w, w), dtype=np.float32) for _ in range(3)]
        
        print(">>> Reconstructing 3D Volumes (FBP, OSEM, MAP)... Please wait.")
        for z in range(d):
            sino = self.sinograms[z].T 
            if np.max(sino) < 1.0: continue
            
            # COR 오프셋 보정
            if CFG['COR_OFFSET'] != 0.0:
                sino = shift(sino, [CFG['COR_OFFSET'], 0], mode='nearest')
                
            v_fbp[z] = np.maximum(iradon(sino, theta=theta, filter_name=CFG['FBP_FILTER'], circle=True), 0)
            
            est = np.ones((w, w), dtype=np.float32)
            sens = iradon(np.ones_like(sino), theta=theta, filter_name=None, circle=True) + 1e-9
            for _ in range(CFG['OSEM_ITER']): est *= (iradon(sino/(radon(est, theta)+1e-9), theta, filter_name=None) / sens)
            v_osem[z] = est
            
            est_m = est.copy()
            for _ in range(CFG['MAP_ITER']):
                upd = iradon(sino/(radon(est_m, theta)+1e-9), theta, filter_name=None, circle=True)
                pen = CFG['MAP_BETA'] * (est_m - gaussian_filter(est_m, sigma=0.8))
                est_m *= (upd / (sens + pen + 1e-9))
            v_map[z] = est_m

        if CFG['FLIP_Z']: v_fbp, v_osem, v_map = [np.flip(v, axis=0) for v in [v_fbp, v_osem, v_map]]
        if CFG['FLIP_Y']: v_fbp, v_osem, v_map = [np.flip(v, axis=1) for v in [v_fbp, v_osem, v_map]]
        self.vols = {'FBP': v_fbp, 'OSEM': v_osem, 'MAP': v_map}
        self.center = np.unravel_index(np.argmax(v_map), (d, w, w))
        print("✅ Reconstruction Complete. Launching Dashboard...")

    def get_metrics(self, th):
        stats = {}
        for algo in ALGO_NAMES:
            vol = self.vols[algo]
            v_max = np.max(vol)
            mask = (vol / (v_max + 1e-9) * 100.0 > th).astype(np.uint8)
            stats[algo] = {'vol_ml': np.sum(mask) * self.meta['vox_ml'], 'mask': mask, 'max': v_max}
        return stats

# ==============================================================================
# 3. Interactive Clinical Dashboard UI
# ==============================================================================
if 'eng' not in globals(): eng = ReconStudio()
sz, sy, sx = eng.meta['shape']
hz, hy, hx = eng.center
asp_z = eng.meta['spacing'][2] / eng.meta['spacing'][0]

def draw_final_dashboard(z, y, x, th, save_path=None):
    data = eng.get_metrics(th)
    fig = plt.figure(figsize=(22, 11))
    gs = fig.add_gridspec(2, 4, width_ratios=[0.6, 1, 1, 1])
    plt.subplots_adjust(wspace=0.15, hspace=0.25, top=0.88, bottom=0.05, right=0.95)
    
    v_unit = eng.meta['vox_ml'] * 1000
    fig.suptitle(f"Thyroid Recon Analysis | Z:{z} Y:{y} | {v_unit:.2f} $mm^3$ | Th: {th}%", 
                 color='#003366', fontsize=24, fontweight='heavy')

    # --- Scouts ---
    ax_sag = fig.add_subplot(gs[0, 0])
    ax_sag.imshow(np.flipud(np.max(eng.vols['MAP'], axis=2)), cmap='gray_r', aspect=asp_z)
    ax_sag.axhline(sz - z - 1, c='#00CC00', lw=2); ax_sag.axvline(y, c='#0066CC', lw=2, ls='--')
    ax_sag.set_title("① Sagittal Scout", color='#444444', fontsize=14); ax_sag.axis('off')

    ax_cor = fig.add_subplot(gs[1, 0])
    ax_cor.imshow(np.flipud(np.max(eng.vols['MAP'], axis=1)), cmap='gray_r', aspect=asp_z)
    ax_cor.axhline(sz - z - 1, c='#00CC00', lw=2); ax_cor.axvline(x, c='#FF3300', lw=2, ls='--')
    ax_cor.set_title("② Coronal Scout", color='#444444', fontsize=14); ax_cor.axis('off')

    # --- Algorithm Matrix ---
    for i, algo in enumerate(ALGO_NAMES):
        col = i + 1
        vol = eng.vols[algo]
        mask = data[algo]['mask']
        vmax = data[algo]['max']
        
        # Axial
        ax_ax = fig.add_subplot(gs[0, col])
        im = ax_ax.imshow(vol[z], cmap=CMAPS[i], vmin=0, vmax=vmax)
        if np.sum(mask[z]) > 0: ax_ax.contour(mask[z], colors='#FFD700', linewidths=1.5)
        ax_ax.set_title(f"{algo}\n({data[algo]['vol_ml']:.2f} ml)", color='#222222', fontsize=16)
        ax_ax.axvline(x, c='#999999', ls=':', alpha=0.8); ax_ax.axhline(y, c='#999999', ls=':', alpha=0.8)
        if col == 1: ax_ax.set_ylabel("AXIAL (Z)", color='#333333', fontsize=14)
        ax_ax.set_xticks([]); ax_ax.set_yticks([])

        cax = ax_ax.inset_axes([0.94, 0.05, 0.04, 0.4])
        cb = plt.colorbar(im, cax=cax)
        cb.ax.tick_params(labelsize=8, color='#333333', labelcolor='#333333')
        cb.outline.set_visible(True); cb.outline.set_edgecolor('#AAAAAA')

        # Coronal
        ax_co = fig.add_subplot(gs[1, col])
        ax_co.imshow(np.flipud(vol[:, y, :]), cmap=CMAPS[i], aspect=asp_z, vmin=0, vmax=vmax)
        if np.sum(np.flipud(mask[:, y, :])) > 0: ax_co.contour(np.flipud(mask[:, y, :]), colors='#FFD700', linewidths=1.5)
        ax_co.set_title(f"{algo} (Coronal)", color='#666666', fontsize=12)
        ax_co.axhline(sz - z - 1, c='#999999', ls=':', alpha=0.8); ax_co.axvline(x, c='#999999', ls=':', alpha=0.8)
        if col == 1: ax_co.set_ylabel("CORONAL (Y)", color='#333333', fontsize=14)
        ax_co.set_xticks([]); ax_co.set_yticks([])

    if save_path: plt.savefig(save_path, dpi=100, bbox_inches='tight', facecolor='white'); plt.close(fig)
    else: plt.show()

# ==============================================================================
# 4. Widget Linking & OpenGATE Data Bridge
# ==============================================================================
st = {'description_width': 'initial'}
sl_z = widgets.IntSlider(value=hz, min=0, max=sz-1, description='Axial Z', style=st)
sl_y = widgets.IntSlider(value=hy, min=0, max=sy-1, description='Coronal Y', style=st)
sl_x = widgets.IntSlider(value=hx, min=0, max=sx-1, description='Sagittal X', style=st)
sl_th = widgets.FloatSlider(value=35, min=5, max=90, description='Threshold %', style=st)
btn_save = widgets.Button(description='📸 Save Image', button_style='info', icon='camera')

out = widgets.Output()

def on_save_click(_):
    fname = os.path.join(CFG['EXPORT_DIR'], f"Recon_Final_Z{sl_z.value}_Th{sl_th.value}.png")
    draw_final_dashboard(sl_z.value, sl_y.value, sl_x.value, sl_th.value, save_path=fname)
    print(f"✅ Saved to {fname}")

def update_ui(z, y, x, th):
    with out:
        clear_output(wait=True)
        draw_final_dashboard(z, y, x, th)

btn_save.on_click(on_save_click)
widgets.interactive_output(update_ui, {'z':sl_z, 'y':sl_y, 'x':sl_x, 'th':sl_th})

display(widgets.VBox([
    widgets.HBox([sl_z, sl_y, sl_x]), 
    widgets.HBox([sl_th, btn_save])
]), out)

"""
# ⚛️ [OpenGATE Voxelized Source Bridge]
# 대시보드 검증 완료 후 아래 주석을 해제하여 GATE 입력용 mhd 파일을 생성하세요.
def export_to_opengate():
    import pydicom
    final_metrics = eng.get_metrics(sl_th.value)
    mask_array = final_metrics['MAP']['mask']
    export_vol = (mask_array * 255).astype(np.uint8)
    
    sitk_img = sitk.GetImageFromArray(export_vol)
    sitk_img.SetSpacing(eng.meta['spacing'])
    
    out_path = f"{CFG['EXPORT_DIR']}/Target_MAP_GATE.mhd"
    sitk.WriteImage(sitk_img, out_path)
    print(f"✅ OpenGATE Input Source 생성 완료: {out_path}")

# export_to_opengate()
"""
